# Data Profiling and Quality Checks

When you receive a new dataset, you should never start modifying it immediately. Real-world data is messy, incomplete, and full of errors. **Data Profiling** is the process of examining your data to understand its structure, content, and quality. 

Think of it as performing a thorough background check on your dataset before you trust it. We use Pandas to ask the data questions: *How big are you? Are you missing any pieces? Do you have any impossible numbers?*

Let's create a deliberately messy dataset to practice our profiling skills.

In [1]:
import pandas as pd
import numpy as np

# Create a deliberately messy dataset
data = {
    'customer_id': [1, 2, 3, 4, 4, 5, 6], # Notice the duplicate ID!
    'age': [25, 32, 45, 28, 28, 999, np.nan], # 999 is impossible, and one is missing (NaN)
    'income': [50000, 65000, 120000, 45000, 45000, 80000, 55000],
    'signup_date': ['2023-01-01', '2023-01-15', '2023-02-20', '2023-03-05', '2023-03-05', '2023-04-10', '2023-05-12'],
    'subscription_type': ['Basic', 'Premium', 'Premium', 'Basic', 'Basic', 'VIP', 'Basic']
}

df = pd.DataFrame(data)
print("✅ Messy dataset loaded into Pandas!")

✅ Messy dataset loaded into Pandas!


# 1. The First Glance (`head`, `shape`, `info`)
Your very first step is to simply look at the size and shape of the data. 

* **`.shape`**: Tells you the exact number of rows and columns.
* **`.head()`**: Shows you the first 5 rows so you can get a visual feel for the data.
* **`.info()`**: The most important command. It tells you the data types (integers, strings, dates) and how many non-null (non-empty) values exist in each column.

In [2]:
print("--- 1. Data Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- 2. First 5 Rows ---")
display(df.head())

print("\n--- 3. Data Info ---")
df.info()

--- 1. Data Shape ---
Rows: 7, Columns: 5

--- 2. First 5 Rows ---


,customer_id,age,income,signup_date,subscription_type
0,1,25.0,50000,2023-01-01,Basic
1,2,32.0,65000,2023-01-15,Premium
2,3,45.0,120000,2023-02-20,Premium
3,4,28.0,45000,2023-03-05,Basic
4,4,28.0,45000,2023-03-05,Basic



--- 3. Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7 non-null      int64  
 1   age                6 non-null      float64
 2   income             7 non-null      int64  
 3   signup_date        7 non-null      object 
 4   subscription_type  7 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 412.0+ bytes


*(Notice in the `.info()` output that 'age' only has 6 non-null values, while everything else has 7. We also see 'signup_date' is currently stored as an 'object' (text), not a proper datetime!)*

# 2. Summary Statistics (`describe`)
The `.describe()` method generates descriptive statistics for all your numerical columns. This is the fastest way to spot wild anomalies or outliers.

In [3]:
print("--- Summary Statistics ---")
display(df.describe())

--- Summary Statistics ---


,customer_id,age,income
count,7.000000,6.000000,7.000000
mean,3.571429,192.833333,65714.285714
std,1.718249,395.002489,26992.062325
min,1.000000,25.000000,45000.000000
25%,2.500000,28.000000,47500.000000
50%,4.000000,30.000000,55000.000000
75%,4.500000,41.750000,72500.000000
max,6.000000,999.000000,120000.000000


*(Look closely at the 'age' column. The `max` age is 999! Because 999 is so large, it has severely dragged the `mean` (average) age up to 188. If we didn't profile our data, we might have trained a model believing our average customer is nearly two centuries old.)*

# 3. Checking for Missing Values (`isnull`)
Missing data (Nulls or NaNs) will completely crash most machine learning algorithms. You must know exactly how many missing values exist in every column so you can plan how to fix them later.

In [4]:
print("--- Missing Values per Column ---")
# .isnull() creates a True/False mask. .sum() adds up all the True values (1s).
missing_values = df.isnull().sum()
print(missing_values)

--- Missing Values per Column ---
customer_id          0
age                  1
income               0
signup_date          0
subscription_type    0
dtype: int64


# 4. Checking for Duplicates (`duplicated`)
Duplicates can skew your analytics and cause models to overfit. We need to see if any exact row repeats itself.

In [5]:
print("--- Duplicate Rows ---")
# Count total duplicates
total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows found: {total_duplicates}")

# Let's actually look at the duplicate row
display(df[df.duplicated(keep=False)]) 
# keep=False shows both the original AND the duplicate so we can compare them

--- Duplicate Rows ---
Total duplicate rows found: 1


,customer_id,age,income,signup_date,subscription_type
3,4,28.0,45000,2023-03-05,Basic
4,4,28.0,45000,2023-03-05,Basic


# 5. Categorical Deep Dive (`value_counts`, `nunique`)
For text/categorical columns, `.describe()` isn't very helpful. Instead, we want to know how many *unique* categories there are, and how the data is distributed among them.

* **`.nunique()`**: Counts how many distinct categories exist.
* **`.value_counts()`**: Shows how many times each category appears.

In [6]:
print("--- Unique Values per Column ---")
print(df.nunique())

print("\n--- Subscription Type Distribution ---")
print(df['subscription_type'].value_counts())

--- Unique Values per Column ---
customer_id          6
age                  5
income               6
signup_date          6
subscription_type    3
dtype: int64

--- Subscription Type Distribution ---
subscription_type
Basic      4
Premium    2
VIP        1
Name: count, dtype: int64


*(We can clearly see that 'Basic' is our most popular tier, and 'VIP' only has one user.)*

## Real-World Use Case or Analogy:
Think of Data Profiling like a **Doctor's Triage Assessment**:

* **`.info()` and `.shape` (Vitals)**: The nurse checks your height, weight, and blood pressure. They are just getting a baseline of who you are and making sure all your basic systems are recording properly.
* **`.describe()` (Lab Results)**: The doctor looks at your blood test. They know the normal human range for cholesterol is between X and Y. If your result is 999, the doctor immediately knows something is severely wrong and investigates further.
* **`.isnull().sum()` (The Questionnaire)**: The doctor reviews your intake forms and notices you left the "Allergies" section completely blank. Before prescribing medicine, they *must* address that missing information. 
* **The Golden Rule**: A doctor would never perform surgery without doing a physical exam first. As a Data Scientist, you should never transform, scale, or model data without doing a `.info()` and `.describe()` profile first!

---